# Search V2 Test Runner

Scratch notebook for exercising the V2 search pipeline one stage at a time on a single query. Currently wired up to run Step 0 (flow routing) in isolation; more stages can be added as they come online.

## Cell 1 — Setup, imports, DB readiness

Run once. Opens the Postgres pool, initializes Redis, and pings Qdrant (idempotent — safe to rerun). Every import used anywhere below lives here.

In [38]:
import sys, time
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Step 0 — flow routing
from search_v2.step_0 import run_step_0

# Shared schemas
from schemas.step_0_flow_routing import Step0Response
from schemas.enums import SearchFlow

# DB
from db.postgres import pool as postgres_pool
from db.qdrant import check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

# DB readiness (idempotent — mirrors test_stage_1_to_4.ipynb)
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


Postgres: pool already open
Redis:    already initialized (ok)
Qdrant:   ok


## Cell 2 — Configuration

Set the query and LLM preset. Uncomment one preset block or assign the three variables manually.

In [39]:
# ---- Query ----
query = "first indiana jones"

# ---- Presets (uncomment one) ----
# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
# provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Shared ----
today = date.today()

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")


Query:    first indiana jones
Provider: gemini
Model:    gemini-3-flash-preview
Kwargs:   {'thinking_config': {'thinking_budget': 0}}
Today:    2026-04-23


## Cell 3 — Step 0: Flow Routing

Runs the raw query through Step 0's classifier. Prints the full `Step0Response` JSON plus a compact summary of which flows fired.

In [ ]:
start = time.perf_counter()
step_0_response, step_0_in_tok, step_0_out_tok = await run_step_0(query)
step_0_elapsed = time.perf_counter() - start

print(f"Step 0 completed in {step_0_elapsed:.2f}s")
print(f"Tokens — input: {step_0_in_tok:,}  output: {step_0_out_tok:,}\n")

etf = step_0_response.exact_title_flow_data
sim = step_0_response.similarity_flow_data

print("FIRING FLOWS")
print("-" * 72)
print(f"  exact_title_flow_data:")
print(f"    should_be_searched:     {etf.should_be_searched}")
print(f"    exact_title_to_search:  {etf.exact_title_to_search!r}")
print(f"  similarity_flow_data:")
print(f"    should_be_searched:     {sim.should_be_searched}")
print(f"    similar_search_title:   {sim.similar_search_title!r}")
print(f"  enable_primary_flow:      {step_0_response.enable_primary_flow}")
print(f"  primary_flow:             {step_0_response.primary_flow.value}")
print()

print("OBSERVATIONS")
print("-" * 72)
print(f"  titles_observed ({len(step_0_response.titles_observed)}):")
for t in step_0_response.titles_observed:
    print(f"    - span={t.span_text!r}")
    print(f"      canonical={t.most_likely_canonical_title!r}")
    print(f"      ambiguity_potential: {t.ambiguity_potential}")
print(f"  qualifiers ({len(step_0_response.qualifiers)}):")
for q in step_0_response.qualifiers:
    print(f"    - {q!r}")
print()

print("FULL JSON")
print("-" * 72)
print(step_0_response.model_dump_json(indent=2))
